In [12]:
import sys
sys.path.append("../src")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from mlflow.tracking import MlflowClient
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss

PROCESSED = Path("../data/processed")
ARTEFACTS = Path("../outputs/models")
FIGURES   = Path("../outputs/figures")

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("home_credit_default_risk")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: sqlite:///../mlflow.db


In [13]:
# --- Artefacts ---
bp_final       = joblib.load(ARTEFACTS / "scorecard_binning.pkl")
lr_final       = joblib.load(ARTEFACTS / "scorecard_model.pkl")
final_features = joblib.load(ARTEFACTS / "scorecard_features_final.pkl")
lgb_final      = joblib.load(ARTEFACTS / "challenger_model.pkl")
lgb_params     = joblib.load(ARTEFACTS / "challenger_params.pkl")
split          = joblib.load(ARTEFACTS / "holdout_split.pkl")

# --- Holdout predictions (already aligned by SK_ID_CURR) ---
pred = pd.read_parquet(PROCESSED / "holdout_predictions.parquet")
y    = pred["TARGET"]
p_sc = pred["p_scorecard"]
p_ch = pred["p_challenger"]

print(f"Holdout: {len(pred):,} rows, bad rate {y.mean():.4f}")
print(f"Scorecard features: {len(final_features)}")

Holdout: 61,503 rows, bad rate 0.0807
Scorecard features: 24


In [14]:
def compute_metrics(y_true, y_prob):
    """Standard credit risk metric set."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    return {
        "auc":     auc,
        "gini":    2 * auc - 1,
        "ks":      np.max(tpr - fpr),
        "brier":   brier_score_loss(y_true, y_prob),
        "logloss": log_loss(y_true, y_prob),
    }

In [15]:
# --- RUN 1: Scorecard, 27 features (pre-pruning, historical) ---
with mlflow.start_run(run_name="scorecard_27_features"):
    mlflow.set_tags({
        "track": "scorecard",
        "stage": "development",
        "model_type": "logistic_regression_woe",
        "note": "Pre-pruning. 2 non-significant + 2 wrong-signed coefficients.",
    })
    mlflow.log_params({
        "n_features": 27,
        "penalty": "none",
        "class_weight": "none",
        "binning": "optbinning BinningProcess, refit per CV fold",
        "min_bin_size": 0.05,
        "max_n_bins": 6,
        "monotonic_trend": "auto_asc_desc",
        "cv_folds": 5,
    })
    mlflow.log_metrics({
        "cv_auc_mean": 0.7470,
        "cv_auc_std":  0.0027,
        "cv_gini":     0.4940,
        "n_nonsignificant": 2,
        "n_wrong_sign": 2,
    })

# --- RUN 2: Scorecard, 24 features (FINAL) ---
with mlflow.start_run(run_name="scorecard_24_features_FINAL"):
    mlflow.set_tags({
        "track": "scorecard",
        "stage": "final",
        "model_type": "logistic_regression_woe",
        "note": "Dropped EXT_SOURCE_1, NONLIVINGAREA_AVG, EXT_2_3_STD. "
                "All coefficients significant and correctly signed.",
    })
    mlflow.log_params({
        "n_features": 24,
        "penalty": "none",
        "class_weight": "none",
        "binning": "optbinning BinningProcess, refit per CV fold",
        "min_bin_size": 0.05,
        "max_n_bins": 6,
        "monotonic_trend": "auto_asc_desc",
        "cv_folds": 5,
        "pdo": 20,
        "base_score": 600,
        "base_odds": 50,
    })

    m = compute_metrics(y, p_sc)
    mlflow.log_metrics({
        "cv_auc_mean": 0.7467,
        "cv_auc_std":  0.0023,
        **{f"holdout_{k}": v for k, v in m.items()},
        "calibration_mad": 0.00375,
        "psi_dev_vs_holdout": 0.00018,
        "n_nonsignificant": 0,
        "n_wrong_sign": 0,
        "score_min": 456,
        "score_max": 657,
    })

    mlflow.sklearn.log_model(lr_final, name="scorecard_model")
    mlflow.log_artifact(str(ARTEFACTS / "scorecard_binning.pkl"))
    mlflow.log_artifact(str(PROCESSED / "scorecard_points_table.csv"))
    mlflow.log_artifact(str(PROCESSED / "score_to_pd_lookup.csv"))
    if (FIGURES / "calibration_curves.png").exists():
        mlflow.log_artifact(str(FIGURES / "calibration_curves.png"))

    print(f"Scorecard logged — holdout AUC {m['auc']:.4f}, KS {m['ks']:.4f}")

Scorecard logged — holdout AUC 0.7424, KS 0.3639


In [16]:
# --- RUN 3: Challenger, unweighted baseline ---
with mlflow.start_run(run_name="challenger_unweighted_baseline"):
    mlflow.set_tags({
        "track": "challenger",
        "stage": "development",
        "model_type": "lightgbm",
        "note": "Untuned baseline. Calibration reference point.",
    })
    mlflow.log_params({
        "n_features": 247, "class_weight": "none",
        "learning_rate": 0.05, "num_leaves": 31,
        "min_child_samples": 100, "colsample_bytree": 0.8,
    })
    mlflow.log_metrics({
        "cv_auc": 0.7852, "cv_logloss": 0.2377, "cv_brier": 0.06604,
    })

# --- RUN 4: Challenger, is_unbalance (REJECTED) ---
with mlflow.start_run(run_name="challenger_is_unbalance_REJECTED"):
    mlflow.set_tags({
        "track": "challenger",
        "stage": "rejected",
        "model_type": "lightgbm",
        "note": "REJECTED. Identical AUC but 2.5x worse calibration. "
                "Unacceptable where PD feeds IFRS 9 provisioning.",
    })
    mlflow.log_params({
        "n_features": 247, "class_weight": "is_unbalance",
        "learning_rate": 0.05, "num_leaves": 31,
    })
    mlflow.log_metrics({
        "cv_auc": 0.7841, "cv_logloss": 0.4976, "cv_brier": 0.16560,
    })

# --- RUN 5: Challenger, Optuna-tuned (FINAL) ---
with mlflow.start_run(run_name="challenger_optuna_tuned_FINAL"):
    mlflow.set_tags({
        "track": "challenger",
        "stage": "final",
        "model_type": "lightgbm",
        "note": "20 Optuna trials. Parameters converged on heavy "
                "regularisation — broad shallow signal.",
    })
    mlflow.log_params({
        "n_features": 247,
        "class_weight": "none",
        "optuna_trials": 20,
        "best_iteration": 1452,
        **lgb_params,
    })

    m = compute_metrics(y, p_ch)
    mlflow.log_metrics({
        "cv_auc": 0.7881,
        **{f"holdout_{k}": v for k, v in m.items()},
        "calibration_mad": 0.00588,
        "psi_dev_vs_holdout": 0.00024,
    })

    mlflow.lightgbm.log_model(lgb_final, name="challenger_model")
    if (PROCESSED / "shap_importance.csv").exists():
        mlflow.log_artifact(str(PROCESSED / "shap_importance.csv"))
    if (FIGURES / "shap_summary.png").exists():
        mlflow.log_artifact(str(FIGURES / "shap_summary.png"))

    print(f"Challenger logged — holdout AUC {m['auc']:.4f}, KS {m['ks']:.4f}")

Challenger logged — holdout AUC 0.7847, KS 0.4365


In [17]:
client = MlflowClient()

# --- Find the final runs by tag ---
runs = mlflow.search_runs(
    experiment_names=["home_credit_default_risk"],
    filter_string="tags.stage = 'final'",
)
print(runs[["run_id", "tags.mlflow.runName", "tags.track"]].to_string(index=False))

sc_run_id = runs[runs["tags.track"] == "scorecard"]["run_id"].iloc[0]
ch_run_id = runs[runs["tags.track"] == "challenger"]["run_id"].iloc[0]

# --- Register both models ---
sc_uri = f"runs:/{sc_run_id}/scorecard_model"
ch_uri = f"runs:/{ch_run_id}/challenger_model"

sc_mv = mlflow.register_model(sc_uri, "home_credit_scorecard")
ch_mv = mlflow.register_model(ch_uri, "home_credit_challenger")

print(f"\nRegistered:")
print(f"  home_credit_scorecard   v{sc_mv.version}")
print(f"  home_credit_challenger  v{ch_mv.version}")

Registered model 'home_credit_scorecard' already exists. Creating a new version of this model...
2026/08/03 12:16:03 WARNING mlflow.tracking._model_registry.fluent: Run with id f7bad23f3ae440618c559c97aec66763 has no artifacts at artifact path 'scorecard_model', registering model based on models:/m-26b67dba719741a39849e847a20dbb61 instead
Created version '2' of model 'home_credit_scorecard'.
Registered model 'home_credit_challenger' already exists. Creating a new version of this model...
2026/08/03 12:16:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 78576abb42034c919a06abedaafc2ef9 has no artifacts at artifact path 'challenger_model', registering model based on models:/m-dc54250dd6b543ad9f1d7d6f1acc53d6 instead


                          run_id           tags.mlflow.runName tags.track
78576abb42034c919a06abedaafc2ef9 challenger_optuna_tuned_FINAL challenger
f7bad23f3ae440618c559c97aec66763   scorecard_24_features_FINAL  scorecard
24172bf234134e23bdf69d8df5875881 challenger_optuna_tuned_FINAL challenger
b1a36a89fbc94c8ea975bf0480afc230   scorecard_24_features_FINAL  scorecard

Registered:
  home_credit_scorecard   v2
  home_credit_challenger  v2


Created version '2' of model 'home_credit_challenger'.


In [18]:
# --- Aliases replace the deprecated stage transitions ---
client.set_registered_model_alias("home_credit_scorecard", "production", sc_mv.version)
client.set_registered_model_alias("home_credit_challenger", "champion", ch_mv.version)

# --- Documentation on the registered models ---
client.update_registered_model(
    name="home_credit_scorecard",
    description=(
        "Regulatory scorecard. Logistic regression on 24 WoE-transformed "
        "features. Holdout AUC 0.7424, Gini 0.4848, KS 0.3639. "
        "Points scale: PDO 20, base score 600 at 50:1 odds, range 456-657. "
        "All coefficients significant and correctly signed. "
        "Supports adverse action reason codes. PRODUCTION MODEL."
    ),
)

client.update_registered_model(
    name="home_credit_challenger",
    description=(
        "LightGBM challenger on 247 features. Holdout AUC 0.7847, "
        "Gini 0.5694, KS 0.4365. Benchmark for quantifying the "
        "interpretability cost of the scorecard (0.0846 Gini). "
        "NOT for production decisions — CODE_GENDER ranks 5th by SHAP, "
        "a protected-attribute fairness exposure."
    ),
)

for name in ["home_credit_scorecard", "home_credit_challenger"]:
    mv = client.get_model_version_by_alias(name, 
         "production" if "scorecard" in name else "champion")
    print(f"{name} v{mv.version} — alias set")

home_credit_scorecard v2 — alias set
home_credit_challenger v2 — alias set


In [19]:
# --- Load models BY ALIAS, exactly as the deployment API will ---
loaded_scorecard  = mlflow.sklearn.load_model("models:/home_credit_scorecard@production")
loaded_challenger = mlflow.lightgbm.load_model("models:/home_credit_challenger@champion")

print("Models loaded from registry by alias.\n")

# --- Verify predictions match the originals exactly ---
bp = joblib.load(ARTEFACTS / "scorecard_binning.pkl")
df_prep = pd.read_parquet(PROCESSED / "preprocessed_train.parquet")
hold_mask = df_prep["SK_ID_CURR"].isin(split["holdout_ids"])
X_check = df_prep[hold_mask][final_features].head(1000)

X_check_woe = bp.transform(X_check)
p_registry = loaded_scorecard.predict_proba(X_check_woe)[:, 1]
p_original = lr_final.predict_proba(X_check_woe)[:, 1]

print(f"Max prediction difference: {np.abs(p_registry - p_original).max():.10f}")
print(f"Identical: {np.allclose(p_registry, p_original)}")

Models loaded from registry by alias.

Max prediction difference: 0.0000000000
Identical: True


In [20]:
with mlflow.start_run(run_name="model_comparison_summary"):
    mlflow.set_tags({
        "track": "comparison",
        "stage": "summary",
        "decision": "Scorecard approved for production. Challenger retained "
                    "as benchmark only.",
    })

    m_sc = compute_metrics(y, p_sc)
    m_ch = compute_metrics(y, p_ch)

    for k in m_sc:
        mlflow.log_metric(f"scorecard_{k}",  m_sc[k])
        mlflow.log_metric(f"challenger_{k}", m_ch[k])

    mlflow.log_metrics({
        "gini_gap":                m_ch["gini"] - m_sc["gini"],
        "auc_gap":                 m_ch["auc"] - m_sc["auc"],
        "scorecard_calib_mad":     0.00375,
        "challenger_calib_mad":    0.00588,
        "bad_rate_90pct_approval_scorecard":  0.0621,
        "bad_rate_90pct_approval_challenger": 0.0576,
        "cost_optimal_scorecard":  336660,
        "cost_optimal_challenger": 299810,
        "cost_approve_all":        496500,
    })

    # Attach all findings documents as artefacts
    for doc in Path("..").glob("PHASE*_FINDINGS.txt"):
        mlflow.log_artifact(str(doc))

    print("\nComparison logged.")
    print(f"  Gini gap: {m_ch['gini'] - m_sc['gini']:.4f}")
    print(f"  Scorecard better calibrated: MAD 0.00375 vs 0.00588")


Comparison logged.
  Gini gap: 0.0847
  Scorecard better calibrated: MAD 0.00375 vs 0.00588
